In [143]:
import pandas as pd
from pathlib import Path





In [83]:
bigfoot_df = pd.read_csv('data/bfro_locations.csv', index_col=False)

In [ ]:
bigfoot_df.shape
bigfoot_df.describe(include='all')
bigfoot_df.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4250 entries, 0 to 4249
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           4250 non-null   int64  
 1   bf_id           4250 non-null   int64  
 2   title           4250 non-null   object 
 3   classification  4250 non-null   object 
 4   timestamp       4250 non-null   object 
 5   latitude        4250 non-null   float64
 6   longitude       4250 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 232.6+ KB


KeyError: 'number'

In [106]:
bigfoot_df.isna().sum()

index             0
bf_id             0
title             0
classification    0
timestamp         0
latitude          0
longitude         0
dtype: int64

In [86]:
bigfoot_df = bigfoot_df.rename(columns={"number": "bf_id"})


In [87]:
bigfoot_df.dtypes

index               int64
bf_id               int64
title              object
classification     object
timestamp          object
latitude          float64
longitude         float64
dtype: object

In [94]:
bigfoot_df.loc[bigfoot_df['bf_id'] == 799]

,index,bf_id,title,classification,timestamp,latitude,longitude
13,13,799,Report 799: Person fishing recounts story of h...,Class A,1978-04-15T12:00:00Z,34.92855,-87.1105


Get The Full Formated Date - Year Month Day 

for non numeric Day entries .. put '01'

use geocoded csv

In [ ]:
bigfoot_geocoded_df = pd.read_csv('data/bfro_reports_geocoded.csv')
bigfoot_geocoded_df['number'] = bigfoot_geocoded_df['number'].astype(int)
bigfoot_geocoded_df.loc[bigfoot_geocoded_df['number'] == 799]


,index,observed,location_details,county,state,season,title,latitude,longitude,date,...,moon_phase,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed
1219,1219,"The main sighting, as I wish to refer to it he...",The main sighting report I leave here happened...,Limestone County,Alabama,Spring,Report 799: Person fishing recounts story of h...,34.92855,-87.1105,1978-04-15,...,0.26,0.0,0.0,NaN,1019.14,Mostly cloudy throughout the day.,5.0,9.91,40.0,2.63


so, it looks like we will need to do a left join from the bigfoot_df to the bigfoot_geocoded_df ... bf_id = number



need the state and city and wx data from the geocoded 

In [114]:
bigfoot_geocoded_df = bigfoot_geocoded_df.rename(columns={"number": "bf_id"})
combined_bigfoot_df = pd.merge(bigfoot_df, bigfoot_geocoded_df[['bf_id', 'date', 'season', 'state', 'geohash',  'temperature_mid', 'precip_type', 'dew_point', 'cloud_cover', 'moon_phase', 'observed']], on='bf_id', how='left')

In [117]:
combined_bigfoot_df.head()


,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,precip_type,dew_point,cloud_cover,moon_phase,observed
0,0,637,Report 637: Campers' encounter just after dark...,Class A,2000-06-16T12:00:00Z,61.5000,-142.9000,2000-06-16,Summer,Alaska,bffmu5xrkw,NaN,rain,42.75,1.00,0.49,My hiking partner and I arrived late to the Ke...
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.1872,-132.7982,1995-05-15,Spring,Alaska,c1c9fne29x,48.595,rain,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,..."
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.2035,-132.8202,2004-02-09,Winter,Alaska,c1cd197r9n,43.490,rain,41.51,0.98,0.62,This incident happened last night just after 1...
3,3,9317,"Report 9317: Driver on Alcan Highway has noon,...",Class A,2004-06-18T12:00:00Z,62.9375,-141.5667,2004-06-18,Summer,Alaska,bg5q496m8b,70.275,NaN,43.18,0.33,0.03,"My daughter and I were traveling to Tok, Alask..."
4,4,13038,Report 13038: Snowmobiler has encounter in dee...,Class A,2004-02-15T12:00:00Z,61.0595,-149.7853,2004-02-15,Winter,Alaska,bdv7re99me,18.665,NaN,17.35,0.16,0.84,I and two of my friends were bored one night s...


In [ ]:
combined_bigfoot_df = combined_bigfoot_df.dropna(subset=['precip_type', 'temperature_mid', 'cloud_cover'])
combined_bigfoot_df.isna().sum()

index               0
bf_id               0
title               0
classification      0
timestamp           0
latitude            0
longitude           0
date                0
season              0
state               0
geohash             0
temperature_mid     0
precip_type         0
dew_point           0
cloud_cover         0
moon_phase          0
observed           14
dtype: int64

In [123]:
combined_bigfoot_df['observed'] = combined_bigfoot_df['observed'].fillna('Unknown')
combined_bigfoot_df.isna().sum()

index              0
bf_id              0
title              0
classification     0
timestamp          0
latitude           0
longitude          0
date               0
season             0
state              0
geohash            0
temperature_mid    0
precip_type        0
dew_point          0
cloud_cover        0
moon_phase         0
observed           0
dtype: int64

In [ ]:
combined_bigfoot_df.to_csv("data/combined_bigfoot_v1.csv", index=False)


get geohash_7, geohash_6, geohash_5 from geohash column

get state_code from state

set date as dateimte
remove redundant datestamp

In [126]:
# bigfoot geohash columns
 
combined_bigfoot_df['geohash_5'] = combined_bigfoot_df['geohash'].str[:5]
combined_bigfoot_df['geohash_6'] = combined_bigfoot_df['geohash'].str[:6]
combined_bigfoot_df['geohash_7'] = combined_bigfoot_df['geohash'].str[:7]

# check the head

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,geohash,temperature_mid,precip_type,dew_point,cloud_cover,moon_phase,observed,geohash_5,geohash_6,geohash_7
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.18720,-132.79820,1995-05-15,Spring,Alaska,c1c9fne29x,48.595,rain,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",c1c9f,c1c9fn,c1c9fne
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.20350,-132.82020,2004-02-09,Winter,Alaska,c1cd197r9n,43.490,rain,41.51,0.98,0.62,This incident happened last night just after 1...,c1cd1,c1cd19,c1cd197
6,6,26604,Report 26604: Daytime sighting of reddish-colo...,Class A,2009-07-15T12:00:00Z,64.89139,-147.81420,2009-07-15,Summer,Alaska,bewchmhgxr,54.370,rain,52.49,0.85,0.77,"It was the month of July, 2009 in Fairbanks Al...",bewch,bewchm,bewchmh
7,7,179,Report 179: Man and witnesses have two seperat...,Class A,1981-09-15T12:00:00Z,32.31435,-85.16235,1981-09-15,Fall,Alabama,djery4dstr,78.455,rain,69.27,0.71,0.55,1981--My first encounter (Sept.): My girlfrien...,djery,djery4,djery4d
8,8,245,"Report 245: Two outdoorsman fishing, loud voca...",Class A,1999-07-15T12:00:00Z,33.28375,-87.32655,1999-07-15,Summer,Alabama,djcvkg4ezu,78.440,rain,70.47,0.38,0.10,A friend and I were fishing on Holt lake in tu...,djcvk,djcvkg,djcvkg4


In [127]:
# mapping dictionary to create state_code column (unique id for state)
# grok4 used to generate dictionary 

state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC'
}

combined_bigfoot_df['state_code'] = combined_bigfoot_df['state'].map(state_to_code)
combined_bigfoot_df['state_code'] = combined_bigfoot_df['state_code'].fillna('Unknown')

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,...,temperature_mid,precip_type,dew_point,cloud_cover,moon_phase,observed,geohash_5,geohash_6,geohash_7,state_code
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.18720,-132.79820,1995-05-15,Spring,Alaska,...,48.595,rain,42.15,0.95,0.54,"I was going for a drive. I had 3 kids with me,...",c1c9f,c1c9fn,c1c9fne,AK
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.20350,-132.82020,2004-02-09,Winter,Alaska,...,43.490,rain,41.51,0.98,0.62,This incident happened last night just after 1...,c1cd1,c1cd19,c1cd197,AK
6,6,26604,Report 26604: Daytime sighting of reddish-colo...,Class A,2009-07-15T12:00:00Z,64.89139,-147.81420,2009-07-15,Summer,Alaska,...,54.370,rain,52.49,0.85,0.77,"It was the month of July, 2009 in Fairbanks Al...",bewch,bewchm,bewchmh,AK
7,7,179,Report 179: Man and witnesses have two seperat...,Class A,1981-09-15T12:00:00Z,32.31435,-85.16235,1981-09-15,Fall,Alabama,...,78.455,rain,69.27,0.71,0.55,1981--My first encounter (Sept.): My girlfrien...,djery,djery4,djery4d,AL
8,8,245,"Report 245: Two outdoorsman fishing, loud voca...",Class A,1999-07-15T12:00:00Z,33.28375,-87.32655,1999-07-15,Summer,Alabama,...,78.440,rain,70.47,0.38,0.10,A friend and I were fishing on Holt lake in tu...,djcvk,djcvkg,djcvkg4,AL


In [132]:
# full_date(datetime), year(int), month(int), dat(int)

combined_bigfoot_df['full_date'] = pd.to_datetime(combined_bigfoot_df['date'], format='%Y-%m-%d', errors='coerce')
combined_bigfoot_df['year']  = combined_bigfoot_df['full_date'].dt.year.astype(int)
combined_bigfoot_df['month'] = combined_bigfoot_df['full_date'].dt.month.astype(int)
combined_bigfoot_df['day']   = combined_bigfoot_df['full_date'].dt.day.astype(int)

combined_bigfoot_df.head()

,index,bf_id,title,classification,timestamp,latitude,longitude,date,season,state,...,moon_phase,observed,geohash_5,geohash_6,geohash_7,state_code,full_date,year,month,day
1,1,2917,Report 2917: Family observes large biped from car,Class A,1995-05-15T12:00:00Z,55.18720,-132.79820,1995-05-15,Spring,Alaska,...,0.54,"I was going for a drive. I had 3 kids with me,...",c1c9f,c1c9fn,c1c9fne,AK,1995-05-15,1995,5,15
2,2,7963,Report 7963: Sasquatch walks past window of ho...,Class A,2004-02-09T12:00:00Z,55.20350,-132.82020,2004-02-09,Winter,Alaska,...,0.62,This incident happened last night just after 1...,c1cd1,c1cd19,c1cd197,AK,2004-02-09,2004,2,9
6,6,26604,Report 26604: Daytime sighting of reddish-colo...,Class A,2009-07-15T12:00:00Z,64.89139,-147.81420,2009-07-15,Summer,Alaska,...,0.77,"It was the month of July, 2009 in Fairbanks Al...",bewch,bewchm,bewchmh,AK,2009-07-15,2009,7,15
7,7,179,Report 179: Man and witnesses have two seperat...,Class A,1981-09-15T12:00:00Z,32.31435,-85.16235,1981-09-15,Fall,Alabama,...,0.55,1981--My first encounter (Sept.): My girlfrien...,djery,djery4,djery4d,AL,1981-09-15,1981,9,15
8,8,245,"Report 245: Two outdoorsman fishing, loud voca...",Class A,1999-07-15T12:00:00Z,33.28375,-87.32655,1999-07-15,Summer,Alabama,...,0.10,A friend and I were fishing on Holt lake in tu...,djcvk,djcvkg,djcvkg4,AL,1999-07-15,1999,7,15


In [ ]:
# kp index 
import kpindex